# 04b_m5_optional_catboost_oof
Backtest OOF của CatBoost mặc định (origin 1829 và 1857, chỉ nhãn Train) cho stacking. Mỗi fold khoảng 35 phút trên CPU 4 lõi.

In [ ]:
import os, pathlib, subprocess, sys, shutil
BRANCH = "m5-end-to-end"
REPO = "https://github.com/haiphamt/IS403.R11-M5-Competition.git"
WORK = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle/working") else "/content")
os.chdir(WORK)  # không đứng trong thư mục repo sắp bị xóa
if (WORK / "repo").exists():
    shutil.rmtree(WORK / "repo")
subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(WORK / "repo")], check=True)
os.chdir(WORK / "repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsforecast", "catboost", "lightgbm", "xgboost", "scikit-learn", "pyarrow", "PyYAML", "optuna"], check=True)
OUT = WORK / "opt"
def run(*cmd):
    subprocess.run([sys.executable, *cmd], check=True)

In [ ]:
run("scripts/backtest_oof.py", "--models", "catboost", "--origins", "1829", "1857", "--n-jobs", str(os.cpu_count()), "--out", str(OUT / "oof_cb"))

In [ ]:
import glob
for d in glob.glob(str(OUT / "**" / "ckpt"), recursive=True):
    shutil.rmtree(d, ignore_errors=True)
print(shutil.make_archive(str(WORK / "m5_optional_oof_catboost"), "zip", str(OUT)))